# Phase 3: Building Mixtral-style Models
## Real-world Applications and Implementation

Let's build a complete Mixtral-style model architecture and explore practical applications.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Dict, Optional, List
import math

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Part 1: Complete Mixtral Transformer Block

A full transformer layer with self-attention and MoE FFN.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Standard multi-head attention (dense, not sparse)."""
    
    def __init__(self, d_model: int, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, d_model = x.shape
        
        # Linear projections
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Combine heads
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, seq_len, d_model)
        
        output = self.W_o(context)
        return output


class MoEFeedForward(nn.Module):
    """Mixture of Experts feed-forward network."""
    
    def __init__(self, d_model: int, num_experts: int = 8, d_hidden: int = None, 
                 top_k: int = 2):
        super().__init__()
        d_hidden = d_hidden or 4 * d_model
        
        self.num_experts = num_experts
        self.top_k = top_k
        
        # Router
        self.router = nn.Linear(d_model, num_experts)
        
        # Experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_hidden),
                nn.ReLU(),
                nn.Linear(d_hidden, d_model),
            )
            for _ in range(num_experts)
        ])
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Dict]:
        batch_size, seq_len, d_model = x.shape
        
        # Router logits
        router_logits = self.router(x)  # (batch, seq, num_experts)
        router_probs = F.softmax(router_logits, dim=-1)
        
        # Select top-k
        expert_weights, expert_indices = torch.topk(router_probs, self.top_k, dim=-1)
        expert_weights = expert_weights / (expert_weights.sum(dim=-1, keepdim=True) + 1e-9)
        
        # Flatten
        x_flat = x.reshape(batch_size * seq_len, d_model)
        output = torch.zeros_like(x_flat)
        
        # Route through experts
        for k in range(self.top_k):
            for expert_idx in range(self.num_experts):
                mask = (expert_indices[:, :, k] == expert_idx).reshape(-1)
                
                if mask.any():
                    expert_output = self.experts[expert_idx](x_flat[mask])
                    weight = expert_weights[:, :, k].reshape(-1)[mask]
                    output[mask] = output[mask] + expert_output * weight.unsqueeze(-1)
        
        output = output.reshape(batch_size, seq_len, d_model)
        
        return output, {'router_probs': router_probs, 'expert_indices': expert_indices}


class TransformerBlock(nn.Module):
    """Complete Transformer block with MoE FFN."""
    
    def __init__(self, d_model: int, num_heads: int = 8, num_experts: int = 8,
                 top_k: int = 2, dropout: float = 0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.moe_ffn = MoEFeedForward(d_model, num_experts, top_k=top_k)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None):
        # Self-attention with residual
        attn_out = self.attention(self.norm1(x), attn_mask)
        x = x + self.dropout(attn_out)
        
        # MoE FFN with residual
        moe_out, moe_stats = self.moe_ffn(self.norm2(x))
        x = x + self.dropout(moe_out)
        
        return x, moe_stats


# Test the block
block = TransformerBlock(d_model=128, num_heads=8, num_experts=8, top_k=2)
x = torch.randn(2, 16, 128)
out, stats = block(x)
print(f"Transformer block test:")
print(f"  Input: {x.shape}")
print(f"  Output: {out.shape}")
print(f"  Parameters: {sum(p.numel() for p in block.parameters())}")

## Part 2: Complete Mixtral-style Language Model

In [ ]:
class MixtralLanguageModel(nn.Module):
    """Complete Mixtral-style language model.
    
    Key features:
    - Embedding layer with position encoding
    - Stack of Transformer blocks with MoE
    - Output head for token prediction
    """
    
    def __init__(self, vocab_size: int, d_model: int = 128, num_layers: int = 4,
                 num_heads: int = 8, num_experts: int = 8, top_k: int = 2,
                 max_seq_len: int = 512, dropout: float = 0.1):
        super().__init__()
        
        self.d_model = d_model
        self.vocab_size = vocab_size
        
        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = self._create_positional_encoding(max_seq_len, d_model)
        
        # Transformer blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, num_experts, top_k, dropout)
            for _ in range(num_layers)
        ])
        
        # Output
        self.norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
    
    def _create_positional_encoding(self, max_len: int, d_model: int) -> torch.Tensor:
        """Create sinusoidal positional encoding."""
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)
    
    def forward(self, input_ids: torch.Tensor, attn_mask: Optional[torch.Tensor] = None):
        """Forward pass through entire model.
        
        Args:
            input_ids: Token IDs (batch, seq_len)
            attn_mask: Attention mask (optional)
            
        Returns:
            logits: Predictions (batch, seq_len, vocab_size)
            moe_stats: MoE statistics from all layers
        """
        batch_size, seq_len = input_ids.shape
        
        # Embeddings
        x = self.token_embedding(input_ids)  # (batch, seq, d_model)
        x = x * math.sqrt(self.d_model)
        
        # Add positional encoding
        pe = self.positional_encoding[:, :seq_len, :].to(x.device)
        x = x + pe
        x = self.dropout(x)
        
        # Transformer blocks
        all_moe_stats = []
        for block in self.transformer_blocks:
            x, moe_stats = block(x, attn_mask)
            all_moe_stats.append(moe_stats)
        
        # Output
        x = self.norm(x)
        logits = self.output_head(x)  # (batch, seq, vocab_size)
        
        return logits, all_moe_stats


# Test the model
vocab_size = 10000
model = MixtralLanguageModel(
    vocab_size=vocab_size,
    d_model=128,
    num_layers=4,
    num_heads=8,
    num_experts=8,
    top_k=2,
    max_seq_len=512
)

input_ids = torch.randint(0, vocab_size, (2, 16))
logits, moe_stats = model(input_ids)

print(f"\nMixtral Language Model:")
print(f"  Input shape: {input_ids.shape}")
print(f"  Output logits shape: {logits.shape}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Parameters per layer: {sum(p.numel() for p in model.transformer_blocks[0].parameters()):,}")

## Part 3: Inference and Generation

In [ ]:
class MixtralInference:
    """Helper for generating text with Mixtral model."""
    
    def __init__(self, model: MixtralLanguageModel):
        self.model = model
        self.device = next(model.parameters()).device
    
    def generate(self, input_ids: List[int], max_length: int = 50,
                 temperature: float = 1.0, top_k: int = 0,
                 top_p: float = 0.9) -> List[int]:
        """Generate tokens using the model.
        
        Args:
            input_ids: Starting token IDs
            max_length: Maximum length to generate
            temperature: Sampling temperature (higher = more random)
            top_k: Top-k sampling (0 = disabled)
            top_p: Nucleus sampling probability
            
        Returns:
            Generated token IDs
        """
        self.model.eval()
        
        current_ids = input_ids.copy()
        
        with torch.no_grad():
            for _ in range(max_length - len(input_ids)):
                # Get predictions
                input_tensor = torch.tensor([current_ids]).to(self.device)
                logits, _ = self.model(input_tensor)
                
                # Get last token's logits
                next_logits = logits[0, -1, :] / temperature
                
                # Apply top-k filtering
                if top_k > 0:
                    indices_to_remove = next_logits < torch.topk(next_logits, top_k)[0][..., -1, None]
                    next_logits[indices_to_remove] = float('-inf')
                
                # Apply top-p (nucleus) filtering
                if top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(next_logits, descending=True)
                    cumsum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    sorted_indices_to_remove = cumsum > top_p
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                    sorted_indices_to_remove[..., 0] = 0
                    indices_to_remove = sorted_indices[sorted_indices_to_remove]
                    next_logits[indices_to_remove] = float('-inf')
                
                # Sample
                probs = F.softmax(next_logits, dim=-1)
                next_token = torch.multinomial(probs, 1).item()
                current_ids.append(next_token)
        
        return current_ids
    
    def compute_perplexity(self, input_ids: torch.Tensor,
                          target_ids: torch.Tensor) -> float:
        """Compute perplexity on a batch.
        
        Args:
            input_ids: Input token IDs
            target_ids: Target token IDs (for computing loss)
            
        Returns:
            Perplexity value
        """
        self.model.eval()
        
        with torch.no_grad():
            logits, _ = self.model(input_ids.to(self.device))
            loss = F.cross_entropy(logits.view(-1, self.model.vocab_size),
                                  target_ids.view(-1).to(self.device))
        
        return torch.exp(loss).item()


# Test inference
inference = MixtralInference(model)

# Generate from random prompt
starter_tokens = [1, 2, 3]  # Random token IDs
generated = inference.generate(starter_tokens, max_length=20, temperature=1.0)

print(f"\nGeneration test:")
print(f"  Starter tokens: {starter_tokens}")
print(f"  Generated sequence (20 tokens): {generated}")
print(f"  Total generated: {len(generated)} tokens")

## Part 4: Performance Analysis

In [ ]:
def benchmark_model(model: MixtralLanguageModel, batch_sizes: List[int],
                   seq_lengths: List[int]) -> Dict:
    """Benchmark model performance across different batch sizes and sequence lengths."""
    model.eval()
    results = {}
    
    with torch.no_grad():
        for batch_size in batch_sizes:
            for seq_len in seq_lengths:
                input_ids = torch.randint(0, model.vocab_size, (batch_size, seq_len))
                input_ids = input_ids.to(next(model.parameters()).device)
                
                # Warm up
                _ = model(input_ids)
                
                # Measure
                import time
                start = time.time()
                for _ in range(5):
                    _ = model(input_ids)
                elapsed = (time.time() - start) / 5
                
                throughput = (batch_size * seq_len) / elapsed
                results[f"B{batch_size}_L{seq_len}"] = {
                    'time_ms': elapsed * 1000,
                    'throughput': throughput,
                }
    
    return results


# Run benchmark
results = benchmark_model(model, batch_sizes=[1, 2, 4], seq_lengths=[16, 32])

print(f"\nPerformance Benchmark:")
print(f"\n{'Config':<20} {'Time (ms)':<15} {'Throughput (tok/s)'}")
print("-" * 50)
for config, metrics in results.items():
    print(f"{config:<20} {metrics['time_ms']:<15.2f} {metrics['throughput']:<15.1f}")

## Part 5: Expert Specialization Analysis

Understanding how experts specialize on different types of inputs.

In [ ]:
def analyze_expert_specialization(model: MixtralLanguageModel, 
                                  input_ids: torch.Tensor,
                                  layer_idx: int = 0) -> Dict:
    """Analyze which experts are specialized for different inputs."""
    model.eval()
    
    with torch.no_grad():
        _, moe_stats = model(input_ids.to(next(model.parameters()).device))
    
    layer_stats = moe_stats[layer_idx]
    router_probs = layer_stats['router_probs']  # (batch, seq, num_experts)
    
    # Analyze preference per token
    avg_probs = router_probs.mean(dim=[0, 1])  # Average across batch and seq
    
    return {
        'expert_preferences': avg_probs.cpu().numpy(),
        'routing_entropy': (-avg_probs * torch.log(avg_probs + 1e-9)).sum().item(),
    }


# Analyze specialization
test_input = torch.randint(0, vocab_size, (2, 16))
spec_analysis = analyze_expert_specialization(model, test_input, layer_idx=0)

print(f"\nExpert Specialization Analysis (Layer 0):")
print(f"  Expert preferences (average probability):")
for i, pref in enumerate(spec_analysis['expert_preferences']):
    print(f"    Expert {i}: {pref:.4f}")
print(f"  Routing entropy: {spec_analysis['routing_entropy']:.4f}")
print(f"    (Lower = more specialized, Higher = more uniform)")

## Summary: Building Production Systems

✅ **Complete Architecture**: Transformer blocks with MoE FFN
✅ **Language Model**: Full autoregressive model with embeddings and output head
✅ **Inference**: Text generation with temperature and nucleus sampling
✅ **Benchmarking**: Performance analysis across different configurations
✅ **Analysis**: Understanding expert specialization

You now have a complete working Mixtral-style implementation ready for:
- Fine-tuning on domain-specific tasks
- Deploying to production
- Experimentation and research
- Integration with larger systems